**Libraries**

In [1]:
import sys
from pathlib import Path

cloned_rep_path=r'C:\Users\laika\OneDrive\Documents\Capstone I All Files\Capstone-DATA4381'
sys.path.insert(0,cloned_rep_path)

from capstone_data.image_loader import load_dataset


In [119]:
from collections import Counter
import pandas as pd
from sklearn.model_selection import train_test_split
import cv2
import numpy as np
import face_recognition
import matplotlib.pyplot as plt
import random
import os


**Dlib and face_recognition**

In [161]:

class GetLandmarks:
    def __init__(self):
        pass
        
    def seperate_classes_split(self, data_path, target_class):
        
        dataset=load_dataset(data_path)
        target_class=target_class.lower()
        
        df=pd.DataFrame(dataset)
        
        # train/val/test split
        
        # First Split
        train_val_df, test_df=train_test_split(
        df, 
        test_size=0.15, 
        random_state=42, 
        stratify=df['class_name']
        )
        # Second Split
        train_df, val_df=train_test_split(
        train_val_df,
        test_size=0.15/0.85,
        random_state=42,
        stratify=train_val_df['class_name']
        )
        
        # Convert train back to list of dictionaries
        train_df=train_df.to_dict(orient='records')
        
        self.paths=[i['path'] for i in train_df if i['class_name']==target_class]
        
        return self.paths
    
    def oversample(self, target_class, image_count):
        
        gap_count=image_count-len(target_class)
        
        duplicates_only=random.choices(target_class, k=gap_count)
        
        oversampled_paths=list(target_class)+duplicates_only
                
        return oversampled_paths, duplicates_only
    
    def undersample(self, target_class, image_count):
        undersampled_paths=random.sample(target_class, k=image_count)
        return undersampled_paths
    
    def upscale_image(self, landmark_image,scale):
        upscale= cv2.resize(landmark_image,scale, interpolation=cv2.INTER_CUBIC)
        return upscale
    
    def right_side(self, landmark_image):
        
        x_right=[]
        y_right=[]
        
        face_landmarks=face_recognition.face_landmarks(landmark_image)
        
        if face_landmarks:
            single_face=face_landmarks[0]
            right_side=['right_eye', 'right_eyebrow']
    
            
            for i in right_side:
                if i in single_face:
                    
                    points=single_face[i]
                    
                    x_right.extend(j[0] for j in points)
                    y_right.extend(j[1] for j in points)
                
        return x_right,y_right

    def left_side(self, landmark_image):
        
        x_left=[]
        y_left=[]
        
        face_landmarks=face_recognition.face_landmarks(landmark_image)
        
        if face_landmarks:
            single_face=face_landmarks[0]
            left_side=['left_eye', 'left_eyebrow']
        
            
            for i in left_side:
                if i in single_face:
            
                    points=single_face[i]
            
                    x_left.extend(j[0] for j in points)
                    y_left.extend(j[1] for j in points)
        
        return x_left,y_left
    
    def mouth_area(self, landmark_image):
        
        x_mouth=[]
        y_mouth=[]
        
        face_landmarks=face_recognition.face_landmarks(landmark_image)
    
        if face_landmarks:
            single_face=face_landmarks[0]
            left_side=['top_lip', 'bottom_lip']
      
    
            for i in left_side:
                if i in single_face:
                    points=single_face[i]
        
                    x_mouth.extend(j[0] for j in points)
                    y_mouth.extend(j[1] for j in points)
        
        return x_mouth,y_mouth

    
    def random_blackout(self, landmark_image, scale):
        
        upscaled_image=self.upscale_image(landmark_image,scale)
        
        facial_features=[self.right_side, self.left_side, self.mouth_area]
        
        chosen_feature=random.choice(facial_features)
        
        x,y=chosen_feature(upscaled_image)
        
        modify_image=upscaled_image.copy()
        
        if x and y:
            
            padding=8
        
            min_x=max(0,min(x)-padding)
            max_x=min(upscaled_image.shape[1], max(x)+padding)
            min_y=max(0,min(y)-padding)
            max_y=min(upscaled_image.shape[0],max(y)+padding)
        
            cv2.rectangle(modify_image, (min_x, min_y), (max_x, max_y), color=(0,0,0), thickness=-1)
        return modify_image
    
    def final_landmarks(self, oversampled_paths, duplicates_only, scale):
        final_images=[]
        
        duplicate_set=set(duplicates_only)
        
        for i in oversampled_paths:
            image=cv2.imread(str(i))
            if i in duplicate_set:
                black_out=self.random_blackout(image, scale)
                final_images.append(black_out)
                
                duplicates_only.remove(i)
                
                # refresh duplicate set
                duplicate_set=set(duplicates_only)
                
            else:
                final_images.append(image)
                
        for j in range(len(final_images)):
            final_images[j]=cv2.resize(final_images[j], (48,48), interpolation=cv2.INTER_AREA)
                
        return final_images
      
        

**Call the class**

In [162]:
get=GetLandmarks()

angry_dataset=get.seperate_classes_split(data_path=r'C:\Users\laika\OneDrive\Documents\Capstone I All Files\MyWorkspaceLP\cleaned_data',target_class='angry')
happy_dataset=get.seperate_classes_split(data_path=r'C:\Users\laika\OneDrive\Documents\Capstone I All Files\MyWorkspaceLP\cleaned_data',target_class='happy')
sad_dataset=get.seperate_classes_split(data_path=r'C:\Users\laika\OneDrive\Documents\Capstone I All Files\MyWorkspaceLP\cleaned_data',target_class='sad')

print(f'Angry Total: {len(angry_dataset)}')
print(f'Happy Total: {len(happy_dataset)}')
print(f'Sad Total: {len(sad_dataset)}')

# Total train is correct
print(f'Train total: {len(angry_dataset)+len(happy_dataset)+len(sad_dataset)}')

Angry Total: 2163
Happy Total: 4673
Sad Total: 845
Train total: 7681


In [165]:
# Oversampling/undersampling

angry_oversampled, angry_duplicates=get.oversample(angry_dataset,4000)
sad_oversampled, sad_duplicates=get.oversample(sad_dataset,4000)
happy_undersampled=get.undersample(happy_dataset,3500)
print(f'Angry Total: {len(angry_oversampled), len(angry_duplicates)}')
print(f'Sad Total: {len(sad_oversampled), len(sad_duplicates)}')
print(f'Happy Total: {len(happy_undersampled)}')



Angry Total: (4000, 1837)
Sad Total: (4000, 3155)
Happy Total: 3500


In [166]:
happy_oversampled, happy_duplicates=get.oversample(happy_undersampled, 4000)

In [167]:
happy_landmarks=get.final_landmarks(happy_oversampled,happy_duplicates.copy(),scale=(400,400))

In [168]:
len(happy_landmarks)

4000

In [169]:
[cv2.imwrite(f"C:/Users/laika/OneDrive/Documents/Capstone I All Files/MyWorkspaceLP/augmented_data/happy_landmarks/img_happy_landmark{i}.jpg", img) for i, img in enumerate(happy_landmarks)]


[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,

In [170]:
angry_landmarks=get.final_landmarks(angry_oversampled,angry_duplicates.copy(),scale=(400,400))

In [ ]:
# sad_landmarks=get.final_landmarks(sad_oversampled,sad_duplicates.copy(),scale=(400,400))

In [171]:
len(angry_landmarks)

4000

In [155]:
len(sad_landmarks)

4000

In [172]:
[cv2.imwrite(f"C:/Users/laika/OneDrive/Documents/Capstone I All Files/MyWorkspaceLP/augmented_data/angry_landmarks/img_angry_landmark{i}.jpg", img) for i, img in enumerate(angry_landmarks)]


[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,

In [ ]:
# [cv2.imwrite(f"C:/Users/laika/OneDrive/Documents/Capstone I All Files/MyWorkspaceLP/augmented_data/sad_landmarks/img_sad_landmark{i}.jpg", img) for i, img in enumerate(sad_landmarks)]


[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,